# Serverless model customization: evaluation

Compare the base model, a Bedrock frontier baseline, and your fine-tuned Model Package.
This track keeps its existing **live** evaluation workflow, model choices, dataset,
generation limits, MLflow integration, and exports.

| Step | Main operation | What you inspect |
| --- | --- | --- |
| 1. Setup | Select the registered model and held-out data | Region, MLflow app, model package group |
| 2. Frontier baseline | Bedrock Converse with local scoring | Accuracy, evidence F1, parse failures |
| 3. Managed scoring | `CustomScorerEvaluator.evaluate()` | Both models' prompts, metrics, and distributions |
| 4. Managed judge | `LLMAsJudgeEvaluator.evaluate()` | Nova Pro scores, coverage, and explanations |
| 5. Compare errors | Tables, charts, and JSON exports | Failure direction and reference-run interpretation |

The core evaluation calls remain below. Supporting implementations are in
[evaluation_utils.py](evaluation_utils.py); the uploaded scorer stays self-contained
in [contractnli_scorer.py](contractnli_scorer.py).

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Setup

In [ ]:
import json
import boto3
from botocore.config import Config
from sagemaker.core.helper.session_helper import Session, get_execution_role
from config import BASE_MODEL_ID, DATASET_PREFIX, resolve_mlflow_app
import contractnli as C
import evaluation_utils as E

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]
REGION, BUCKET = sess.boto_region_name, sess.default_bucket()
BEDROCK_REGION = REGION
FRONTIER = "global.anthropic.claude-sonnet-4-6"
EVALUATOR_MODEL = "amazon.nova-pro-v1:0"
base_model_id = BASE_MODEL_ID
model_package_group_name = E.model_package_group_name(base_model_id)
sm = boto3.client("sagemaker", region_name=REGION)
mlflow_app_arn = resolve_mlflow_app(sm)
s3 = boto3.client("s3", region_name=REGION)
print(f"{REGION} | {model_package_group_name} | MLflow: {mlflow_app_arn}")

In [ ]:
C.ensure_dataset("./data")
test_docs, labels = C.load("test")
CHECKLIST = list(labels.keys())

# Evaluate on the whole held-out split so every number in this notebook is
# comparable with the managed pipeline, which always uses the full registered
# dataset. Lower this if you want a faster pass; expect a couple of points of noise.
EVAL_N = len(test_docs)
EVAL_DOCS = test_docs[:EVAL_N]

print(f"{len(EVAL_DOCS)} contracts x {len(CHECKLIST)} checklist items "
      f"= {len(EVAL_DOCS) * len(CHECKLIST)} decisions")

In [ ]:
SCORER_OUTPUT = f"s3://{BUCKET}/contractnli-scorer-eval"
JUDGE_OUTPUT = f"s3://{BUCKET}/contractnli-judge-eval"
results = E.EvaluationResults(
    s3=s3, sm=sm, bucket=BUCKET, documents=EVAL_DOCS, checklist=CHECKLIST,
    base_model_id=base_model_id, frontier_model=FRONTIER,
    scorer_output=SCORER_OUTPUT, judge_output=JUDGE_OUTPUT)

### What we measure

Scoring flattens each model's answer into one row per **(contract, checklist item)**
pair, and every metric is computed over those rows.

| Metric | Definition | Why |
|---|---|---|
| `accuracy` | fraction of rows where the predicted label equals gold | the obvious metric, and the misleading one |
| `contradiction_f1` | F1 for the `Contradiction` class, one-vs-rest | the rare class (~10% of rows) and the expensive mistake: a clause that *conflicts* with a requirement |
| `evidence_f1` | overlap of cited and gold span numbers, micro-averaged over spans | cannot be reached from the label distribution — the spans are specific to this document |
| `parse_failures` | contracts whose output was not valid JSON | a model that cannot produce parseable output is unusable regardless of accuracy |

Two details worth knowing. `evidence_f1` sums true/false positives and negatives
across all rows *before* dividing, so a row citing five spans counts five times as
much as a row citing one. And rows whose gold answer cites nothing — the
`NotMentioned` ones, 43% of rows on this split — are excluded from it rather than counted as
trivially correct.

`EVAL_N` changes the local frontier pass, not the registered managed-evaluation dataset.

## 2. Establish the frontier baseline

The utility retains Converse request construction, temperature fallback, throttling
retries, six concurrent workers, parsing, and local scoring. The managed
model comparisons in the next section use the Model Package selected from this track's config.

In [ ]:
bedrock = boto3.client(
    "bedrock-runtime", region_name=BEDROCK_REGION,
    config=Config(read_timeout=300, retries={"total_max_attempts": 3}))
frontier_metrics, frontier_rows = E.evaluate_with_bedrock(
    EVAL_DOCS, FRONTIER, client=bedrock, labels=labels, workers=6)
print(FRONTIER, frontier_metrics)
results.save_json("frontier_metrics", frontier_metrics)
results.save_json("frontier_rows", frontier_rows)

## 3. Run managed statistical scoring

The scorer reads `model_response`, not the `response` reference field. Keep the
gold-versus-lazy-answer sanity check before registering it. Local evidence F1 pools
spans across the corpus; the managed scorer averages per-contract F1.

In [ ]:
SHOW_SCORER_SOURCE = False
if SHOW_SCORER_SOURCE:
    E.show_scorer_source()
E.check_scorer(EVAL_DOCS[0])

In [ ]:
from sagemaker.ai_registry.air_constants import REWARD_FUNCTION
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.evaluator import Evaluator

SCORER_NAME = "contractnli-scorer"
try:
    scorer = Evaluator.get(name=SCORER_NAME)
    print("reusing registered scorer")
except Exception:
    scorer = Evaluator.create(name=SCORER_NAME, type=REWARD_FUNCTION,
                              source="contractnli_scorer.py", role=role,
                              sagemaker_session=sess, wait=True)
    scorer.refresh()
    print("registered scorer")

test_dataset = DataSet.get(name=f"{DATASET_PREFIX}-test")
resp = sm.list_model_packages(
    ModelPackageGroupName=model_package_group_name, SortBy="CreationTime",
    SortOrder="Descending", MaxResults=1)
assert resp["ModelPackageSummaryList"], "no model packages found - run notebook 2 first"
model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print(f"fine-tuned model: {model_package_arn}")

### Submit the job

These are the original evaluator settings. `evaluate_base_model=True` scores both models with the same data and scorer; no endpoint is needed.

In [ ]:
from sagemaker.train.evaluate import CustomScorerEvaluator

scorer_eval = CustomScorerEvaluator(
    evaluator=scorer,
    dataset=test_dataset,
    model=model_package_arn,
    model_package_group=model_package_group_name,
    s3_output_path=SCORER_OUTPUT,
    evaluate_base_model=True,
    mlflow_resource_arn=mlflow_app_arn,
    sagemaker_session=sess,
    role=role,
)

scorer_eval.hyperparameters.max_new_tokens = 8192
scorer_eval.hyperparameters.max_model_len = 24000
print("generation settings:",
      {k: v for k, v in scorer_eval.hyperparameters.to_dict().items()
       if k in ("max_new_tokens", "max_model_len", "temperature")})

scorer_execution = scorer_eval.evaluate()
print("launched:", scorer_execution.arn)
E.wait_for_pipeline(sm, scorer_execution.arn)

### Verify prompts and inspect results

These calls retain the checks on both inference paths, execution-specific result
selection, restart recovery, scorer-failure warnings, and JSON exports.

In [ ]:
SCORER_EXECUTION = results.resolve_execution(
    globals().get("scorer_execution"), "CustomScorerEvaluation")
results.verify_prompts(SCORER_EXECUTION)
base_scored, tuned_scored = results.scorer_results(SCORER_EXECUTION)
results.compare_models(base_scored, frontier_metrics, tuned_scored)
E.plot_statistical_comparison(
    base_scored, frontier_metrics, tuned_scored, FRONTIER.split(".")[-1],
    num_contracts=len(EVAL_DOCS), checklist_items=len(CHECKLIST))
base_details, tuned_details = results.scorer_distributions(
    base_scored, tuned_scored, SCORER_EXECUTION)

<details>
<summary>How to interpret the comparison and per-contract distributions</summary>

### All three models

The scorer and the local pass compute the *same definitions* but average them
differently: the managed scorer runs per record and averages, so its evidence figure
is a mean of per-contract F1, while the local pass pools every span before dividing.
They land within about a point on this dataset — close enough to compare, different
enough not to present as one number. Keep that in mind reading the table below: the
fine-tuned row comes out slightly *ahead* of the frontier row, by 0.2 accuracy and 1.8
evidence-F1. Since the two are not computed by the same averaging, treat that as the
small model having reached parity, not as a ranking it won.

### The same numbers as a chart

The table above is the record; the chart is what you put in front of someone else.
Note the scale: the managed scorer reports fractions (0-1) and the local `metrics()`
pass reports percentages (0-100), so the frontier numbers are divided by 100 to put
all three models on one axis. Mixing the two silently is the easiest way to publish a
chart that is wrong by a factor of a hundred.

### Behind the averages

Those four aggregate numbers are means over 123 contracts, and a mean can hide the
shape it came from: 60% accuracy is a very different model if every contract scores
0.6 than if half score 1.0 and half score 0.2.

The pipeline writes the per-record scores too — a Parquet file next to the aggregates,
one row per contract — so we can plot the distribution rather than take the mean on
faith. The cell asserts that the rows reproduce the aggregates, which is also a check
that we are reading the right run's file.

</details>

## 4. Run LLM-as-a-Judge

Nova Pro judges the existing full test dataset for both models. All three custom
prompts and rating scales remain in `E.build_judge_metrics()`:
`EvidenceSupportsVerdict`, `CarveOutHandling`, and `ChecklistCompleteness`.
The returned definitions can still be inspected and edited before submission.

> **Warning — two separate IAM requirements.** The judge runs on Amazon Bedrock, so
> the SageMaker execution role needs **both**:
>
> 1. a **trust relationship** allowing `bedrock.amazonaws.com` to assume it, and
> 2. an **identity-based policy** granting `bedrock:CreateEvaluationJob` (plus
>    `GetEvaluationJob`, `ListEvaluationJobs`, `StopEvaluationJob`) and
>    `bedrock:InvokeModel` on the evaluator model's inference profile.
>
> The trust relationship alone is not enough, and `AmazonSageMakerFullAccess` does
> **not** grant `bedrock:CreateEvaluationJob`. If the second is missing, the inference
> steps succeed and the run fails ~25 minutes later at the metrics step with
> `AccessDeniedException ... not authorized to perform: bedrock:CreateEvaluationJob on
> resource: .../inference-profile/us.amazon.nova-pro-v1:0`.
>
> See [Bedrock evaluation permissions](https://docs.aws.amazon.com/bedrock/latest/userguide/judge-service-roles.html).

In [ ]:
custom_metrics = E.build_judge_metrics()
custom_metrics_json = json.dumps(custom_metrics)
print("Judge:", EVALUATOR_MODEL)
print("Metrics:", [m["customMetricDefinition"]["name"] for m in custom_metrics])

In [ ]:
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

judge = LLMAsJudgeEvaluator(
    model=model_package_arn,
    model_package_group=model_package_group_name,
    evaluator_model=EVALUATOR_MODEL,
    dataset=test_dataset,
    custom_metrics=custom_metrics_json,   # a JSON string, not a list
    s3_output_path=JUDGE_OUTPUT,
    evaluate_base_model=True,
    mlflow_resource_arn=mlflow_app_arn,
    sagemaker_session=sess,
)
judge_execution = judge.evaluate()
print("launched:", judge_execution.arn)
E.wait_for_pipeline(sm, judge_execution.arn)
judge_execution.show_results(limit=5, offset=0, show_explanations=True)

This is the longest wait in the lab: **45-55 minutes**, against 15-20 for the
statistical job. Both models are inferred over all 123 contracts, then every response
is read again by the judge against three metrics — two full passes over the same text
rather than one.

Each judged record carries a full contract in the prompt (~3,200 tokens on average, up
to ~9,800 for the longest), so this job reads a few million tokens across both models. Check Bedrock pricing for the evaluator
model before running it on a larger test set.

### Scores and coverage

Judge scores stay on a 0–1 scale. Missing scores remain missing. Both charts and both models' JSON exports are retained.

In [ ]:
JUDGE_EXECUTION = results.resolve_execution(
    globals().get("judge_execution"), "LLMAJEvaluation")
judged = results.judge_results(JUDGE_EXECUTION)

<details>
<summary>Reading judge scores</summary>

### Plotting the judge

The judge writes one JSONL record per contract, each carrying a score and the
reasoning behind it. Two things about those scores decide how they should be plotted:

- **They are already normalised to 0-1.** A metric declared on a 0-3 rating scale
  comes back as 0, 0.333, 0.667 or 1.0 — so `EvidenceSupportsVerdict` at 0.41 means
  1.2 out of 3, not 41% of anything. Reading it as a percentage overstates the model.
- **A record can go unscored.** When the judge cannot read an answer it returns
  `null`, not 0. Those stay `NaN` here rather than being counted as zeros, because a
  metric averaged over 38 contracts is a weaker claim than one averaged over 123, and
  a model whose answer did not parse would otherwise be charged twice — once by the
  metric it failed, once by the zero. The coverage is printed under each bar so the
  difference stays visible.

### Reading the judge results

Read each bar against the coverage printed under it (`base scored N/123`). The judge
returns nothing when it cannot decide, and it declines on different records for each
model — so where coverage is low and unequal, the delta compares two different subsets
of contracts rather than the same ones before and after.


</details>

## 5. Compare error directions

The table and consequence-colored chart show which mistakes matter for contract review.

In [ ]:
E.show_error_directions(frontier_rows, FRONTIER)

<details>
<summary>Interpretation and takeaways from the original lesson</summary>

`show_explanations=True` is the reason to use a judge at all: each score comes with
the model's reasoning, so a disagreement with the statistical scorer is something you
can read and adjudicate rather than guess about.

---

## Putting the two together

| | Statistical scorer | LLM-as-a-Judge |
|---|---|---|
| Reproducible | identical every run | varies between runs |
| Cost | free | pays per token, on long prompts |
| Judges semantic citation validity | no — span numbers only | yes |
| Can be gamed | no | prompt injection is a real risk |
| Good for | regression gates, CI, tracking a model over time | understanding *why* a model is wrong |

Use the statistical metrics as the gate and the judge to explain the residual. If the
judge scores `EvidenceSupportsVerdict` high while `evidence_f1` is low, the model is
citing valid clauses the annotator did not pick — which is a labelling artefact, not
a model failure.

### Where the errors are

Error *direction* matters more than error rate. A false `Entailment` tells a reviewer
a protection exists when it does not; a false `NotMentioned` merely misses one.

The cell above plots the frontier model, whose dominant error is
`NotMentioned -> Entailment` — claiming a protection the contract never grants. That
direction, not the raw rate, is what decides whether a contract review system can
ship: a false "yes, you're covered" is a different production risk than "missed one".

Compare the same breakdown for your own model, and for the base model too.

### Takeaways

1. **Lead with a metric that cannot be reached without reading the document** —
   here, `evidence_f1` and `contradiction_f1`. Accuracy alone rewards a model that
   has learned the label distribution: answering `NotMentioned` to every item scores
   43% accuracy on this test split for free, and scores **exactly zero** on evidence.
2. **Check the parse rate before reading any average, and then move on.** An answer
   that does not parse scores zero on every item, so `json_valid` is the denominator
   the other metrics have to be read against.
3. **Statistical scoring and a judge answer different questions.** One gates, the
   other explains.
4. **Prompting is not always the cheaper answer.** Here it made the small model
   measurably worse at the decision that mattered.

Continue to **notebook 4** to serve the model on a SageMaker real-time endpoint.

</details>

Continue to **notebook 4** to deploy and clean up the endpoint.